Imports and config (always run this first)

In [1]:
import os
import numpy as np
import sys
sys.path.append('/mnt/md0/tempFolder/samAnderson/unet-gnn/')
sys.path.append('/mnt/md0/tempFolder/samAnderson/unet-gnn/functions/')
from config import feature_config as cfg
from postprocessing import PostProcessor
%load_ext autoreload
%autoreload 2

# Create the postprocessing object
p = PostProcessor(first=cfg.first)

Run ablation to determine feature significance

In [ ]:
'''
To get ablation results, use get_ablation.py
'''

Get differences in LBAGs per cohort

In [2]:
# Define hierarchy (higher - lower)
hierarchy = ['AD', 'CN']

# Load ablation (delta LBAG) dictionaries per cohort
delta_lbag_dict = {}
for name, cohort_cfg in cfg.cohort_dict.items():
    delta_lbag_dict[name] = np.load(
        f"{cohort_cfg.array_path}ablation_dict.npy",
        allow_pickle=True
    ).item()

# Order cohorts
ordered = [c for c in hierarchy if c in delta_lbag_dict]

# Get feature list
features = list(next(iter(delta_lbag_dict.values())).keys())

# Compute pairwise differences
for i in range(len(ordered)):
    for j in range(i + 1, len(ordered)):

        name0 = ordered[i]
        name1 = ordered[j]

        dict0 = delta_lbag_dict[name0]
        dict1 = delta_lbag_dict[name1]

        comparison_name = f'{name0}-{name1}'
        cohort_vis_path = f'{cfg.cross_cohort_path}{comparison_name}/visualizations/ablation/'
        cohort_array_path = f'{cfg.cross_cohort_path}{comparison_name}/arrays/ablation/'

        os.makedirs(cohort_vis_path, exist_ok=True)
        os.makedirs(cohort_array_path, exist_ok=True)

        # Feature-wise delta, delta LBAGs
        for feature in features:

            delta0 = dict0[feature]  # (nodes,)
            delta1 = dict1[feature]

            # Difference in feature importance between cohorts
            delta_diff_map = delta0 - delta1

            # Save
            np.save(f'{cohort_array_path}{feature}_delta_lbag_diff.npy', delta_diff_map)

            # Plot
            plot_base = f'{cohort_vis_path}{feature}_delta_lbag_diff'
            p.generate_cortical_plot(
                delta_diff_map,
                medial_present=False,
                save_to=plot_base,
                abs_limits=2.5,
                cbar_present = False # will make as single bar for all plots later
            )

Get network and region-level saliencies for each feature

In [3]:
import pandas as pd

# Define the difference pairs
diff_pairs = {
    'AD-CN': ('AD', 'CN')
}

dfs_region = []
dfs_network = []

# Load dict for each cohort
all_cohorts_dict = {}
for cohort in cfg.cohort_dict.keys():
    all_cohorts_dict[cohort] = np.load(f'{cfg.cohort_dict[cohort].array_path}ablation_dict.npy',
                                       allow_pickle=True
                                       ).item()

# Loop over features
for feature in cfg.features:
    
    # Create feature dict
    feature_dict = {}
        
    # Add each cohort's corresponding feature
    for cohort in cfg.cohort_dict.keys():
        feature_dict[cohort] = all_cohorts_dict[cohort][feature]
    
    # Build per-cohort dfs
    df_region = cfg.build_location_df(
        data_dict=feature_dict,
        postprocessing_object=p,
        per='region',
        medial_present=False,
        add_diffs=True,
        diff_pairs=diff_pairs
    )

    df_network = cfg.build_location_df(
        data_dict=feature_dict,
        postprocessing_object=p,
        per='network',
        medial_present=False,
        add_diffs=True,
        diff_pairs=diff_pairs
    )

    # Add feature column
    df_region['feature'] = feature
    df_network['feature'] = feature

    # Append
    dfs_region.append(df_region)
    dfs_network.append(df_network)

# Combine all cohorts
df_region = pd.concat(dfs_region, ignore_index=True)
df_network = pd.concat(dfs_network, ignore_index=True)

# Build region to lobe mapping
region_to_lobe = {
    region: lobe
    for lobe, regions in cfg.region_to_lobe_dict.items()
    for region in regions
}

# Map regions to lobes
df_lobe = df_region.copy()
df_lobe['lobe'] = df_lobe['region'].map(region_to_lobe)
df_lobe = df_lobe.dropna(subset=['lobe'])
df_lobe = (
    df_lobe
    .groupby(['feature', 'lobe'])
    .mean(numeric_only=True)
    .reset_index()
)
cols = [c for c in df_lobe.columns if c != 'feature'] + ['feature']
df_lobe = df_lobe[cols]

# Ensure output directory exists
out_dir = f'{cfg.cross_cohort_path}all/arrays/'
os.makedirs(out_dir, exist_ok=True)

# Save dfs
df_region.to_csv(f'{out_dir}ablations_region.csv')
df_lobe.to_csv(f'{out_dir}ablations_lobe.csv')
df_network.to_csv(f'{out_dir}ablations_network.csv')